# Baku's bus network from AYNA's open map: a first look

The dataset is a snapshot of the public map API of AYNA, Baku's land transport agency, scraped on 23 October 2026: 208 bus routes, 3,841 stops, the ordered stops of each route direction and the shape of each direction. It is the raw data behind the paper *How efficient are Baku's bus routes? An open-data measurement and a fleet-neutral redesign* and the repository [ismatsamadov/route_efficiency](https://github.com/ismatsamadov/route_efficiency) (archived at [doi.org/10.5281/zenodo.23604370](https://doi.org/10.5281/zenodo.23604370)).

This notebook loads the tables, draws the network and the stops, and shows two things the raw data does **not** tell you without care: the declared lengths and durations are not measurements, and a route's two directions have different stop IDs. The repository's `busnet` package repairs the defects before measuring anything; here we only look.

In [ ]:
import glob
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

matches = glob.glob("/kaggle/input/**/ayna.db", recursive=True)   # wherever Kaggle mounted the dataset
print("found:", matches)
DB = matches[0]
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
tables = {name: pd.read_sql(f'SELECT * FROM "{name}"', con)
          for (name,) in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
for name, df in tables.items():
    print(f"{name:13s} {len(df):7,d} rows   {', '.join(df.columns)}")

## The routes

One row per route. `route_length_km` is the **round-trip** length and `duration_min` is simply that length divided by 0.6 on every route, so neither is a measurement of a one-way trip; the plot below makes the second point visible.

In [ ]:
buses = tables["buses"]
print(buses[["number", "carrier", "first_point", "last_point", "route_length_km", "duration_min", "working_zone"]].head(10).to_string(index=False))
print()
print("carriers:", buses["carrier"].nunique(), "| routes by working zone:")
print(buses["working_zone"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(buses["route_length_km"], buses["duration_min"], s=12)
ax.set_xlabel("route_length_km (round trip)"); ax.set_ylabel("duration_min")
ax.set_title("duration_min is length ÷ 0.6 on every route")
plt.show()

## The stops and the network on a map

`stops` has one row per stop *per route direction*: the same physical stop has a different `stop_id` for each direction and often for each route, which is why a journey planner on this data needs walking links between nearby stops. `route_points` holds every shape as a sequence of coordinates; drawing them all gives the network.

In [ ]:
stops, pts = tables["stops"], tables["route_points"]

# the city and the peninsula; a few intercity routes run off the edge
LAT, LON = (40.27, 40.62), (49.58, 50.36)
fig, ax = plt.subplots(figsize=(11, 7))
for (bus_id, direction), g in pts.sort_values(["bus_id", "direction", "seq"]).groupby(["bus_id", "direction"]):
    ax.plot(g["lon"], g["lat"], lw=0.5, color="#2a78d6", alpha=0.6)
ax.scatter(stops["lon"], stops["lat"], s=1.5, color="#0b0b0b", alpha=0.5, label="stops")
ax.set_xlim(LON); ax.set_ylim(LAT); ax.set_aspect(1 / np.cos(np.radians(40.4)))
ax.set_title("All 398 route directions (blue) and 3,841 stops (black)")
ax.legend(loc="lower right"); plt.show()

## How direct is a route?

A quick version of the paper's first measure: the length of a direction's shape along its points, divided by the straight line between its first and last stop. The paper counts a route as *direct* up to 1.4, *indirect* to 2, *very indirect* to 3 and a *loop* above that or when the ends are less than 1 km apart; it takes each route's less direct direction. The raw shapes have defects (some are stored backwards or filed under the wrong direction), so these quick numbers differ a little from the paper's 29 / 98 / 55 / 19.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

rows = []
for (bus_id, direction), g in pts.sort_values(["bus_id", "direction", "seq"]).groupby(["bus_id", "direction"]):
    lat, lon = g["lat"].to_numpy(), g["lon"].to_numpy()
    along = haversine_km(lat[:-1], lon[:-1], lat[1:], lon[1:]).sum()
    straight = haversine_km(lat[0], lon[0], lat[-1], lon[-1])
    rows.append((bus_id, direction, along, straight))
shapes = pd.DataFrame(rows, columns=["bus_id", "direction", "path_km", "straight_km"])
shapes["circuity"] = np.where(shapes["straight_km"] >= 1, shapes["path_km"] / shapes["straight_km"], np.inf)

per_route = shapes.groupby("bus_id")["circuity"].max().rename("circuity").reset_index().merge(buses[["bus_id", "number"]])
bins = [0, 1.4, 2, 3, np.inf]
per_route["class"] = pd.cut(per_route["circuity"], bins, labels=["direct", "indirect", "very indirect", "loop"])
print(per_route["class"].value_counts().reindex(["direct", "indirect", "very indirect", "loop"]).to_string())
print()
print("least direct routes (raw shapes, before repair):")
print(per_route.sort_values("circuity", ascending=False).head(8)[["number", "circuity"]].to_string(index=False))

## Where to go from here

- The repository repairs the defects (shapes filed under the wrong direction, stored backwards, with swapped coordinates), measures directness, overlap, detours, stop spacing and coverage, builds a door-to-door travel model checked against BakuBus boardings, and searches for a fleet-neutral redesign: https://github.com/ismatsamadov/route_efficiency
- The paper describes the measures and the results: preprint on engrXiv (submission 8632), under review at *Public Transport*.
- A route planner built on the same data: https://www.birjob.com/aletler/avtobus-marsrutu

The data is AYNA's and carries no published licence; it is redistributed as a research snapshot with attribution.